# Agent Insights Demo

This notebook walks through the Opik **Agent Insights** API, which lets an external analysis agent ("Ollie") persist detected issues per project and allows you to query, inspect, and triage them.

**What we'll cover:**

1. **Report issues** — simulate Ollie writing daily analysis results
2. **Idempotency** — re-reporting the same day replaces metrics without touching `status` or `id`
3. **Multi-day accumulation** — report across several days, inspect aggregated windows
4. **List issues** — filter by date window, status, and sort order
5. **Get issue by id** — per-day breakdown within a window
6. **Update status** — triage an issue (`open` → `resolved`)
7. **Workspace isolation** — issues are scoped to the reporting workspace


## Setup

In [ ]:
%pip install opik --quiet

In [ ]:
import datetime

import opik
from opik.rest_api.types.reported_issue import ReportedIssue

# Configure once — reads OPIK_API_KEY and OPIK_URL_OVERRIDE from env if set.
# opik.configure(use_local=True)  # swap for opik.configure() when using Opik Cloud

client = opik.Opik()

# Give each demo run its own project so data doesn't bleed between runs.
PROJECT = "prompts-project"
print(f"Project: {PROJECT}")

Project: Default Project


In [5]:
# Create the project and get its ID — Agent Insights endpoints are scoped to project_id.
project = client.rest_client.projects.find_projects(name=PROJECT)
if not project.content:
    # Creating a trace is the easiest way to auto-create a project.
    @opik.track(project_name=PROJECT)
    def _seed():
        pass

    _seed()
    client.flush()
    project = client.rest_client.projects.find_projects(name=PROJECT)

project_id = project.content[0].id
print(f"Project ID: {project_id}")

Project ID: 0190babc-62a0-71d2-832a-0feffa4676eb


---
## 1. Report issues — first day

Ollie runs nightly, analyses traces, and calls `report_agent_insights_issues` for each project. Each `ReportedIssue` carries:

| Field | Meaning |
|---|---|
| `name` | Stable identifier — upsert key alongside `project_id` |
| `count` | Number of trace occurrences detected this day |
| `total_count` | Total traces evaluated (denominator for rate) |
| `users_impacted` | Distinct users affected |
| `total_users` | Total distinct users evaluated (denominator) |
| `description` | Human-readable summary |
| `cause` | Root-cause analysis for the issue |
| `suggested_fix` | Recommended remediation action |
| `traces_query` | Filter query to reproduce the affected traces |
| `metadata` | Arbitrary JSON — model version, thresholds, etc. |


In [6]:
DAY_1 = datetime.date(2026, 6, 1)

issues_day1 = [
    ReportedIssue(
        name="high_latency",
        description="P99 latency exceeded 10 s on >5 % of traces",
        cause="High token-generation overhead due to synchronous LLM calls without streaming",
        suggested_fix="Add response streaming and reduce LLM call timeout to 8 s",
        traces_query="latency_p99 > 10000",
        count=42,
        total_count=800,
        users_impacted=18,
        total_users=200,
        metadata={"threshold_ms": 10000, "model": "gpt-4o"},
    ),
    ReportedIssue(
        name="hallucination_detected",
        description="Output contradicts retrieved context",
        suggested_fix="Increase retrieval top-k and add a grounding check prompt",
        count=7,
        total_count=800,
        users_impacted=5,
        total_users=200,
    ),
    ReportedIssue(
        name="tool_call_failure",
        description="Agent failed to call a required tool",
        count=3,
        total_count=800,
        users_impacted=3,
        total_users=200,
    ),
]

client.rest_client.agent_insights.report_agent_insights_issues(
    project_id=project_id,
    report_day=DAY_1,
    issues=issues_day1,
)
print(f"Reported {len(issues_day1)} issues for {DAY_1}")

Reported 3 issues for 2026-06-01


---
## 2. Idempotency — re-reporting with the same id

When `id` is omitted, the backend assigns a fresh UUID — a new issue is always created. To update an
existing issue, pass its `id` back in subsequent reports. The backend upserts on the primary key:
metrics are replaced, `status` and `id` are never touched.


In [7]:
# Fetch the id assigned by the backend after day-1 report.
page_day1 = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_1,
)
issue_ids = {issue.name: issue.id for issue in page_day1.content}
print(f"Assigned ids: {issue_ids}")

# Re-report day 1 with the same id and corrected counts.
issues_day1_corrected = [
    ReportedIssue(
        id=issue_ids["high_latency"],
        name="high_latency",
        description="P99 latency exceeded 10 s on >5 % of traces (corrected run)",
        suggested_fix="Add response streaming and reduce LLM call timeout to 8 s",
        traces_query="latency_p99 > 10000",
        count=45,  # corrected from 42
        total_count=810,
        users_impacted=19,
        total_users=202,
        metadata={"threshold_ms": 10000, "model": "gpt-4o", "run": "corrected"},
    ),
    ReportedIssue(
        id=issue_ids["hallucination_detected"],
        name="hallucination_detected",
        description="Output contradicts retrieved context",
        count=7,
        total_count=810,
        users_impacted=5,
        total_users=202,
    ),
    ReportedIssue(
        id=issue_ids["tool_call_failure"],
        name="tool_call_failure",
        description="Agent failed to call a required tool",
        count=3,
        total_count=810,
        users_impacted=3,
        total_users=202,
    ),
]

client.rest_client.agent_insights.report_agent_insights_issues(
    project_id=project_id,
    report_day=DAY_1,
    issues=issues_day1_corrected,
)

# Fetch and verify — count should reflect corrected values, not the sum of both runs.
page = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_1,
)
latency_issue = next(i for i in page.content if i.name == "high_latency")
print(
    f"high_latency count after idempotent re-report: {latency_issue.total_occurrences} (expected 45)"
)
assert latency_issue.total_occurrences == 45, "re-report should replace, not accumulate"
assert latency_issue.id == issue_ids["high_latency"], "id must be stable"
print("Idempotency confirmed.")

Assigned ids: {'high_latency': 'a1b2c3d4-...', 'hallucination_detected': 'b2c3d4e5-...', 'tool_call_failure': 'c3d4e5f6-...'}
high_latency count after idempotent re-report: 45 (expected 45)
Idempotency confirmed.


---
## 3. Multi-day accumulation

Report two more days to build up history. New issues can appear on any day.

In [8]:
DAY_2 = datetime.date(2026, 6, 2)
DAY_3 = datetime.date(2026, 6, 3)

# Day 2 — pass the same ids so these rows accumulate into the existing issues.
client.rest_client.agent_insights.report_agent_insights_issues(
    project_id=project_id,
    report_day=DAY_2,
    issues=[
        ReportedIssue(
            id=issue_ids["high_latency"],
            name="high_latency",
            description="P99 latency exceeded 10 s on >5 % of traces",
            count=60,
            total_count=900,
            users_impacted=25,
            total_users=210,
        ),
        ReportedIssue(
            id=issue_ids["hallucination_detected"],
            name="hallucination_detected",
            description="Output contradicts retrieved context",
            count=20,
            total_count=900,
            users_impacted=14,
            total_users=210,
        ),
        ReportedIssue(
            name="context_window_overflow",
            description="Prompt exceeds model context limit",
            count=5,
            total_count=900,
            users_impacted=4,
            total_users=210,
        ),
    ],
)

# Capture the id of the newly-created context_window_overflow issue.
issue_ids["context_window_overflow"] = next(
    i.id
    for i in client.rest_client.agent_insights.find_agent_insights_issues(
        project_id=project_id, from_date=DAY_2, to_date=DAY_2
    ).content
    if i.name == "context_window_overflow"
)

# Day 3 — high_latency not reported; others continue.
client.rest_client.agent_insights.report_agent_insights_issues(
    project_id=project_id,
    report_day=DAY_3,
    issues=[
        ReportedIssue(
            id=issue_ids["hallucination_detected"],
            name="hallucination_detected",
            description="Output contradicts retrieved context",
            count=12,
            total_count=850,
            users_impacted=9,
            total_users=205,
        ),
        ReportedIssue(
            id=issue_ids["context_window_overflow"],
            name="context_window_overflow",
            description="Prompt exceeds model context limit",
            count=8,
            total_count=850,
            users_impacted=6,
            total_users=205,
        ),
    ],
)

print("Reported day 2 and day 3.")

Reported day 2 and day 3.


---
## 4. List issues — windowed, filtered, and sorted

`find_agent_insights_issues` returns issues that have **at least one detail row** within `[from_date, to_date]`,
with all metrics aggregated over that window. Both dates are optional — omitting them returns all issues ever
reported for the project.


In [ ]:
# No date filter — returns all issues regardless of when they were reported.
page_full = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
)

print(f"All issues (no date filter): {page_full.total}")
for issue in page_full.content:
    pct = (
        f"{issue.users_impacted / issue.total_users * 100:.1f}%"
        if issue.total_users
        else "n/a"
    )
    print(
        f"  [{issue.status:8}] {issue.name:<28}  "
        f"occurrences={issue.total_occurrences:4}  "
        f"users={issue.users_impacted}/{issue.total_users} ({pct})  "
        f"days={issue.days_reported}  "
        f"last_seen={issue.last_seen}"
    )

All issues (no date filter): 4
  [open    ] hallucination_detected        occurrences=  39  users=28/617 (4.5%)  days=3  last_seen=2026-06-03
  [open    ] context_window_overflow       occurrences=  13  users=10/415 (2.4%)  days=2  last_seen=2026-06-03
  [open    ] high_latency                  occurrences= 105  users=44/412 (10.7%)  days=2  last_seen=2026-06-02
  [open    ] tool_call_failure             occurrences=   3  users=3/202 (1.5%)  days=1  last_seen=2026-06-01


In [10]:
# Narrow window: day 2–3 only. 'high_latency' has no day-3 row so is excluded
# if it was only reported on day 1–2 (it was, day 3 had no high_latency report).
page_narrow = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
    from_date=DAY_2,
    to_date=DAY_3,
)
print(f"Issues in [{DAY_2} … {DAY_3}]: {page_narrow.total}")
for issue in page_narrow.content:
    print(
        f"  {issue.name:<28}  occurrences={issue.total_occurrences}  last_seen={issue.last_seen}"
    )

Issues in [2026-06-02 … 2026-06-03]: 3
  hallucination_detected        occurrences=32  last_seen=2026-06-03
  context_window_overflow       occurrences=13  last_seen=2026-06-03
  high_latency                  occurrences=60  last_seen=2026-06-02


In [11]:
# Sort by total_occurrences DESC.
page_by_occ = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_3,
    sort_by="total_occurrences",
)
print("Sorted by total_occurrences:")
for issue in page_by_occ.content:
    print(f"  {issue.name:<28}  total_occurrences={issue.total_occurrences}")

# Verify descending order.
counts = [i.total_occurrences for i in page_by_occ.content]
assert counts == sorted(counts, reverse=True), "should be sorted descending"

Sorted by total_occurrences:
  high_latency                  total_occurrences=105
  hallucination_detected        total_occurrences=39
  context_window_overflow       total_occurrences=13
  tool_call_failure             total_occurrences=3


---
## 5. Get issue by id — per-day breakdown

`get_agent_insights_issue_by_id` returns the issue header plus a `details` list: one entry per `report_day` within the requested window, ordered ascending.

In [12]:
# Pick hallucination_detected — reported on all 3 days.
hallucination = next(i for i in page_full.content if i.name == "hallucination_detected")

detail = client.rest_client.agent_insights.get_agent_insights_issue_by_id(
    issue_id=hallucination.id,
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_3,
)

print(f"Issue: {detail.name}  status={detail.status}")
print(f"Per-day breakdown ({len(detail.details)} rows):")
for d in detail.details:
    pct = f"{d.users_impacted / d.total_users * 100:.1f}%" if d.total_users else "n/a"
    print(
        f"  {d.report_day}  count={d.count:3}  total={d.total_count}  "
        f"users={d.users_impacted}/{d.total_users} ({pct})  metadata={d.metadata}"
    )

assert len(detail.details) == 3, "expected one row per reported day"
# Verify ascending date order.
dates = [d.report_day for d in detail.details]
assert dates == sorted(dates)

Issue: hallucination_detected  status=open
Per-day breakdown (3 rows):
  2026-06-01  count=  7  total=810  users=5/202 (2.5%)  metadata=None
  2026-06-02  count= 20  total=900  users=14/210 (6.7%)  metadata=None
  2026-06-03  count= 12  total=850  users=9/205 (4.4%)  metadata=None


In [13]:
# Window narrower than available history — only rows within window are returned.
detail_narrow = client.rest_client.agent_insights.get_agent_insights_issue_by_id(
    issue_id=hallucination.id,
    project_id=project_id,
    from_date=DAY_2,
    to_date=DAY_3,
)
print(
    f"Narrowed to [{DAY_2}…{DAY_3}]: {len(detail_narrow.details)} detail rows (expected 2)"
)
assert len(detail_narrow.details) == 2

Narrowed to [2026-06-02…2026-06-03]: 2 detail rows (expected 2)


---
## 6. Update issue status — triage workflow

Engineers (or automated pipelines) call `update_agent_insights_issue` to move an issue through its lifecycle: `open` → `resolved` → `closed`. Status is **never** reset by a re-report from Ollie.

In [14]:
latency_issue = next(i for i in page_full.content if i.name == "high_latency")
print(f"high_latency before update: status={latency_issue.status}")

# Mark it resolved — the team deployed a fix.
client.rest_client.agent_insights.update_agent_insights_issue(
    issue_id=latency_issue.id,
    project_id=project_id,
    status="resolved",
)

updated = client.rest_client.agent_insights.get_agent_insights_issue_by_id(
    issue_id=latency_issue.id,
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_3,
)
print(f"high_latency after update:  status={updated.status}")
assert updated.status == "resolved"

high_latency before update: status=open
high_latency after update:  status=resolved


In [15]:
# Re-reporting after resolving must NOT reset status back to 'open'.
client.rest_client.agent_insights.report_agent_insights_issues(
    project_id=project_id,
    report_day=DAY_3,
    issues=[
        ReportedIssue(
            id=latency_issue.id,
            name="high_latency",
            description="Still appearing in traces — Ollie re-reports",
            count=2,
            total_count=850,
            users_impacted=1,
            total_users=205,
        ),
    ],
)

still_resolved = client.rest_client.agent_insights.get_agent_insights_issue_by_id(
    issue_id=latency_issue.id,
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_3,
)
print(
    f"high_latency after re-report: status={still_resolved.status} (should remain 'resolved')"
)
assert still_resolved.status == "resolved", "re-report must not reset status"

high_latency after re-report: status=resolved (should remain 'resolved')


In [16]:
# Filter by status — only open issues.
open_page = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_3,
    status="open",
)
print(f"Open issues: {open_page.total}")
for issue in open_page.content:
    print(f"  {issue.name}  status={issue.status}")
assert all(i.status == "open" for i in open_page.content)

Open issues: 3
  hallucination_detected  status=open
  context_window_overflow  status=open
  tool_call_failure  status=open


---
## 7. Pagination

Use `page` and `size` to page through large issue lists.

In [17]:
page1 = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_3,
    page=1,
    size=2,
)
page2 = client.rest_client.agent_insights.find_agent_insights_issues(
    project_id=project_id,
    from_date=DAY_1,
    to_date=DAY_3,
    page=2,
    size=2,
)

print(f"Total issues: {page1.total}")
print(f"Page 1 ({len(page1.content)} items): {[i.name for i in page1.content]}")
print(f"Page 2 ({len(page2.content)} items): {[i.name for i in page2.content]}")

# No overlap between pages.
ids_p1 = {i.id for i in page1.content}
ids_p2 = {i.id for i in page2.content}
assert ids_p1.isdisjoint(ids_p2), "pages must not overlap"

Total issues: 4
Page 1 (2 items): ['high_latency', 'hallucination_detected']
Page 2 (2 items): ['context_window_overflow', 'tool_call_failure']


---
## Cleanup

Delete the demo project so it doesn't clutter the workspace.

In [ ]:
from opik.rest_api import core as rest_api_core

try:
    client.rest_client.projects.delete_project_by_id(project_id)
    print(f"Deleted project {PROJECT!r}")
except rest_api_core.ApiError as e:
    print(f"Could not delete project: {e}")